In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from keras.preprocessing import image
from keras.models import Sequential, Model
from keras.layers import Dense, LeakyReLU, BatchNormalization, Reshape, Flatten, Input
from keras.models import load_model
from keras.optimizers import Adam

2023-12-15 23:13:57.575837: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-12-15 23:13:57.696721: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-15 23:13:57.696761: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-15 23:13:57.697251: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-15 23:13:57.755311: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-12-15 23:13:57.756410: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

In [7]:
# Specify the path to the folder containing your images
dataset_path = "dataset"

# Get a list of all image files in the folder
image_files = [f for f in os.listdir(dataset_path) if f.endswith('.jpg')]  # Update the extension based on your image format

# Initialize an empty array to store the images
images = []
counter = 0

# Load and preprocess each image
for file_name in image_files:
    file_path = os.path.join(dataset_path, file_name)

    # Load the image using keras.preprocessing.image.load_img
    img = image.load_img(file_path)  # Adjust the target_size based on your requirements

    # Convert the image to a numpy array
    img_array = image.img_to_array(img)

    # Preprocess the image (adjust as needed)
    img_array = img_array.astype(np.float32) / 255

    # Flatten the image and add it to the list
    images.append(img_array)

# Convert the list of images to a numpy array
X_train = np.array(images)
print("X_train shape:", X_train.shape)

# Reshape the images
X_train = X_train.reshape(X_train.shape[0], -1)  # -1 automatically computes the remaining dimension

print("X_train shape:", X_train.shape)

X_train shape: (4854, 128, 128, 3)
X_train shape: (4854, 49152)


In [3]:
def train_gan(generator, discriminator, gan, X_train, epochs=1, batch_size=128):
    batch_count = X_train.shape[0] // batch_size

    for e in range(epochs):
        for _ in range(batch_count):
            noise = np.random.normal(0, 1, size=[batch_size, 100])
            generated_images = generator.predict(noise)
            image_batch = X_train[np.random.randint(0, X_train.shape[0], size=batch_size)]
            discriminator.trainable = True
            d_loss_real = discriminator.train_on_batch(image_batch, np.ones(batch_size))
            d_loss_fake = discriminator.train_on_batch(generated_images, np.zeros(batch_size))
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            noise = np.random.normal(0, 1, size=[batch_size, 100])
            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, np.ones(batch_size))

        print(f"Epoch {e}/{epochs}, D Loss: {d_loss}, G Loss: {g_loss}")

        if (e + 1) % 10 == 0:
            plot_generated_images(generator, e)

In [4]:
def plot_generated_images(generator, epoch, examples=10, dim=(1, 10), figsize=(10, 1)):
    noise = np.random.normal(0, 1, size=[examples, 100])
    generated_images = generator.predict(noise)
    generated_images = generated_images.reshape(examples, 128, 128, 3)
    plt.figure(figsize=figsize)

    for i in range(generated_images.shape[0]):
        plt.subplot(dim[0], dim[1], i + 1)
        plt.imshow((generated_images[i] * 255).astype(np.uint8), cmap='viridis', interpolation="nearest")
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'gan_generated_image_epoch_{epoch+7000}.png')
    plt.show()

In [5]:
# Load Generator
loaded_generator = load_model("generator_model.h5")

# Load Discriminator
loaded_discriminator = load_model("discriminator_model.h5")

# Load GAN
loaded_gan = load_model("gan_model.h5")

# Compile the loaded models with the same optimizer and loss
loaded_generator.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))
loaded_discriminator.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))
loaded_gan.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))

In [8]:
# Continue training
train_gan(loaded_generator, loaded_discriminator, loaded_gan, X_train, epochs=30, batch_size=50)

2/2 [==============================] - 0s 54ms/step
Epoch 0/30, D Loss: 0.33553052321076393, G Loss: 9.208989126818778e-11
2/2 [==============================] - 0s 59ms/step


KeyboardInterrupt: 

In [7]:
loaded_generator.save("generator_model.h5")
loaded_discriminator.save("discriminator_model.h5")
loaded_gan.save("gan_model.h5")